# Plan d.vii -- Post-Estimation Diagnostics

Validates that Model A (static OLS) and Model B (the capped ARDL(1,1) / UECM carried forward
from step vi) satisfy the assumptions their standard errors and significance tests rely on, in
the exact order given in the requirements doc, and applies the Newey-West HAC correction if the
autocorrelation/heteroskedasticity checks flag a problem in Model A.

See `docs/2_plan/analysis/vii_post_estimation_diagnostics.md` for the full spec.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (from step i)
- `modeling_path_decision.csv` (from step v)
- `ardl_capped_1_1_bounds_test.csv` (from step vi -- the bounds F-test is restated here, not
  recomputed)

Step vi does not persist fitted model objects to disk, so this notebook re-estimates Model A
(`statsmodels.api.OLS`) and the capped ARDL(1,1) (`statsmodels.tsa.ardl.UECM`) exactly as step vi
did, then sanity-checks the refit against step vi's saved coefficient/fit-stat CSVs before running
any diagnostics.

**Outputs** (`outputs/`):
- `diagnostics_model_a_ols.csv`, `diagnostics_model_b_ardl_capped.csv`,
  `diagnostics_model_diff_ols.csv`
- `cusum_cusumsq_model_b_status.csv`
- `hac_remediation_status.csv`, `model_a_hac_coefficients.csv` (Newey-West HAC-robust SEs for
  Model A, reported side by side with the original OLS SEs unconditionally -- see Step 6)
- `ardl_capped_1_1_long_run_hac_comparison.csv`, `ardl_capped_1_1_short_run_hac_comparison.csv`
  (Model B's HAC-robust SEs, side by side with the originals -- see Step 6b)
- `model_diff_hac_coefficients.csv` (the first-differenced OLS's HAC-robust SEs, side by side
  with the originals -- see Step 6c; this model was redesignated the primary model for H1/H2
  inference based on this notebook's and step vi's evidence)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats as sstats
from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    het_breuschpagan,
    linear_reset,
    recursive_olsresiduals,
)
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.tools import add_constant
from statsmodels.tsa.ardl import UECM

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
DECISION_IN = OUTPUT_DIR / "modeling_path_decision.csv"
MODEL_A_COEF_IN = OUTPUT_DIR / "model_a_static_ols_coefficients.csv"
MODEL_A_FIT_IN = OUTPUT_DIR / "model_a_static_ols_fit_stats.csv"
CAPPED_BOUNDS_IN = OUTPUT_DIR / "ardl_capped_1_1_bounds_test.csv"

DIAG_A_OUT = OUTPUT_DIR / "diagnostics_model_a_ols.csv"
DIAG_B_OUT = OUTPUT_DIR / "diagnostics_model_b_ardl_capped.csv"
DIAG_DIFF_OUT = OUTPUT_DIR / "diagnostics_model_diff_ols.csv"
CUSUM_STATUS_OUT = OUTPUT_DIR / "cusum_cusumsq_model_b_status.csv"
HAC_STATUS_OUT = OUTPUT_DIR / "hac_remediation_status.csv"
HAC_COEF_OUT = OUTPUT_DIR / "model_a_hac_coefficients.csv"
HAC_B_LONG_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_long_run_hac_comparison.csv"
HAC_B_SHORT_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_short_run_hac_comparison.csv"
MODEL_DIFF_HAC_OUT = OUTPUT_DIR / "model_diff_hac_coefficients.csv"

REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}

CAPPED_P, CAPPED_Q = 1, 1  # the spec carried forward from step vi (Step B8)
BOUNDS_CASE = 3


def fmt_p(p):
    return None if p is None or (isinstance(p, float) and np.isnan(p)) else round(float(p), 4)


def sig_decision(p, alpha=0.05):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "reject H0 (p < 0.05)" if p < alpha else "fail to reject H0 (p >= 0.05)"


def dw_bounds_verdict(dw, dL, dU):
    # Two-sided Durbin-Watson bounds-test decision (Savin-White critical values).
    if dw < dL:
        return "positive autocorrelation (DW < dL)"
    if dw > 4 - dL:
        return "negative autocorrelation (DW > 4-dL)"
    if dU <= dw <= 4 - dU:
        return "no autocorrelation (dU <= DW <= 4-dU)"
    return "inconclusive (DW falls in the bounds-test gray zone)"

## Step 0 -- Reload inputs and re-estimate every model diagnosed in this notebook

Re-fits Model A (static OLS on levels), the capped ARDL(1,1) UECM, and the first-differenced OLS
exactly as step vi did, then checks each refit against step vi's saved coefficients so a silent
spec drift between notebooks would fail loudly rather than quietly change the diagnostics below.

**Note on "primary model" terminology:** `modeling_path_decision.csv` (step v) and step vi both
record ARDL/UECM (Model B) as the primary model under Branch B, with the first-differenced OLS
reported as a comparison model, not the primary one -- see `research_plan.md` lines 20, 159. The
first-differenced OLS is diagnosed here regardless, per this plan's own "Decisions & flags"
section: "Diagnostics are run on all estimated models from step vi (Model A always; Model B and
the first-differenced OLS if Branch B applies)" -- Branch B applies, so it belongs in this
notebook's scope independent of which model is labeled primary.

In [2]:
frame = pd.read_csv(FRAME_IN).set_index("year")
assert frame.shape[0] == 35, f"expected 35-row analysis frame, got {frame.shape[0]}"

eri = frame["eri"].rename("ERI")
X = frame[list(REGRESSORS.values())].rename(columns={v: k for k, v in REGRESSORS.items()})

decision = pd.read_csv(DECISION_IN).iloc[0]
print(f"Branch from step v: {decision['branch']}")
print(f"models_to_estimate: {decision['models_to_estimate']}")

# --- Model A: static OLS on levels ---
design_a = add_constant(X, has_constant="add")
model_a = sm.OLS(eri, design_a).fit()

saved_a_coefs = pd.read_csv(MODEL_A_COEF_IN).set_index("term")["coef"]
refit_diff = (model_a.params.reindex(saved_a_coefs.index) - saved_a_coefs).abs().max()
assert refit_diff < 1e-6, f"Model A refit does not match step vi's saved coefficients (max abs diff={refit_diff})"
print(f"Model A refit OK -- n={int(model_a.nobs)}, df_resid={int(model_a.df_resid)}, matches step vi's saved coefficients (max abs diff={refit_diff:.2e})")

Branch from step v: B
models_to_estimate: OLS on first-differenced variables (primary model for H1/H2 inference, redesignated 2026-07-17 -- see Addendum below); static OLS on levels (literal-thesis comparison, unchanged); ARDL(1,1) bounds testing (secondary/exploratory -- bounds test inconclusive at both the grid-searched and leanest specifications; Breusch-Godfrey LM(2) flags uncorrected serial correlation at 5%).
Model A refit OK -- n=35, df_resid=28, matches step vi's saved coefficients (max abs diff=7.11e-17)


In [3]:
# --- Model B: capped ARDL(1,1) / UECM, the spec carried forward from step vi ---
uecm_model = UECM(eri, lags=CAPPED_P, exog=X, order=CAPPED_Q, trend="c")
uecm_capped = uecm_model.fit()

saved_b_ect = pd.read_csv(OUTPUT_DIR / "ardl_capped_1_1_error_correction_term.csv")["coef"].iloc[0]
ect_diff = abs(uecm_capped.params["ERI.L1"] - saved_b_ect)
assert ect_diff < 1e-6, f"Model B refit does not match step vi's saved error-correction term (diff={ect_diff})"
print(f"Model B (capped ARDL(1,1)/UECM) refit OK -- n={int(uecm_capped.nobs)}, df_resid={int(uecm_capped.df_resid)}, matches step vi's saved ECT (diff={ect_diff:.2e})")

Model B (capped ARDL(1,1)/UECM) refit OK -- n=34, df_resid=20, matches step vi's saved ECT (diff=0.00e+00)


### OLS-equivalent reconstruction of Model B

`UECMResults` objects are not `RegressionResultsWrapper` instances, so statsmodels' regression
diagnostics helpers (`acorr_breusch_godfrey`, `linear_reset`, `recursive_olsresiduals`) raise
`NotImplementedError`/`TypeError` when called on them directly -- they were built for plain OLS
results. `UECM` is internally estimated by OLS on a transformed design matrix (`ERI.L1` and the
six regressors' first lags as levels, plus each regressor's contemporaneous first difference), so
that exact transformed design matrix is pulled out of the fitted `UECM` model
(`uecm_model._x`, `uecm_model._y`) and re-run through `sm.OLS` directly. The coefficients are
verified to match `uecm_capped.params` exactly (not just approximately) before this OLS-equivalent
object is used for any diagnostic test below -- it is a re-expression of the same fitted model,
not a different model.

In [4]:
exog_equiv = pd.DataFrame(
    uecm_model._x, columns=uecm_capped.model.exog_names, index=X.index[-uecm_model._x.shape[0]:]
)
endog_equiv = pd.Series(uecm_model._y, name="D.ERI", index=exog_equiv.index)
model_b_ols_equiv = sm.OLS(endog_equiv, exog_equiv).fit()

max_abs_diff = (model_b_ols_equiv.params - uecm_capped.params).abs().max()
assert max_abs_diff == 0.0, f"OLS-equivalent reconstruction of Model B does not exactly match UECM (max abs diff={max_abs_diff})"
print(f"OLS-equivalent reconstruction of Model B verified exact (max abs coef diff = {max_abs_diff}) -- n={int(model_b_ols_equiv.nobs)}, df_resid={int(model_b_ols_equiv.df_resid)}, k_params={len(model_b_ols_equiv.params)}")

diag_rows_a = []
diag_rows_b = []
diag_rows_diff = []

OLS-equivalent reconstruction of Model B verified exact (max abs coef diff = 0.0) -- n=34, df_resid=20, k_params=14


In [5]:
# --- First-differenced OLS: the comparison spec that removes the I(1)-regressor,
# spurious-regression risk by differencing (see step vi) ---
diff_frame = pd.concat([eri, X], axis=1).diff().dropna()
diff_eri = diff_frame["ERI"]
diff_X = diff_frame.drop(columns="ERI")
design_diff = add_constant(diff_X, has_constant="add")
model_diff = sm.OLS(diff_eri, design_diff).fit()

saved_diff_coefs = pd.read_csv(OUTPUT_DIR / "model_b_first_differenced_ols_coefficients.csv").set_index("term")["coef"]
diff_refit_diff = (model_diff.params.reindex(saved_diff_coefs.index) - saved_diff_coefs).abs().max()
assert diff_refit_diff < 1e-6, f"First-differenced OLS refit does not match step vi's saved coefficients (max abs diff={diff_refit_diff})"
print(f"First-differenced OLS refit OK -- n={int(model_diff.nobs)}, df_resid={int(model_diff.df_resid)}, matches step vi's saved coefficients (max abs diff={diff_refit_diff:.2e})")

First-differenced OLS refit OK -- n=34, df_resid=27, matches step vi's saved coefficients (max abs diff=6.55e-17)


## Step 1 -- Autocorrelation

- **Model A (OLS):** Durbin-Watson statistic, tested against the **exact Savin-White (1977)
  bounds table** for n=35, k=6 regressors (excluding the constant) at the 5% level, rather than a
  rough rule-of-thumb -- the naive "close to 2" read is not precise enough to resolve a DW value
  that is not obviously near 2.
  Source: Savin, N. E., and White, K. J. (1977), "The Durbin-Watson Test for Serial Correlation
  with Extreme Sample Sizes or Many Regressors," *Econometrica*, 45(8), 1989-1996. Table values
  (5% significance points of d_L, d_U) taken from the reproduction at the University of York,
  Department of Mathematics, Historical Statistical Tables archive:
  <https://www.york.ac.uk/depts/maths/histstat/tables/durbinwatson.pdf> (page 3, n=35 row,
  "Lambda=6" column -- Lambda is that table's notation for the number of regressors excluding the
  constant, matching Model A's 6 regressors exactly).
- **Model B (ARDL/UECM):** Durbin-Watson is unreliable with a lagged dependent variable
  (`ERI.L1`), which the conditional-ECM form includes by construction -- the **Breusch-Godfrey LM
  test** is used instead. The model's own `(p,q) = (1,1)` lag structure implies lag order 1 as the
  primary test; lag order 2 is also run as an explicit sensitivity check on whether the lag-1
  result (p close to 0.05) holds up at a longer lag.

In [6]:
# Savin & White (1977) 5% bounds, n=35, k=6 regressors excluding the constant -- see markdown
# above for the full citation. Hardcoded because it is a fixed statistical-table lookup for this
# exact (n, k), not a value derived from the data; the assert below guards against silent drift
# if the design matrix ever changes shape.
DW_N, DW_K = int(model_a.nobs), X.shape[1]
assert (DW_N, DW_K) == (35, 6), (
    f"Savin-White dL/dU below are hardcoded for n=35, k=6; got n={DW_N}, k={DW_K} -- look up the "
    "correct table row before proceeding."
)
DW_DL, DW_DU = 1.160, 1.803

dw_stat = durbin_watson(model_a.resid)
dw_verdict = dw_bounds_verdict(dw_stat, DW_DL, DW_DU)

dw_interp = (
    f"DW={dw_stat:.4f} vs. Savin-White 5% bounds dL={DW_DL}, dU={DW_DU} (n=35, k=6) -> {dw_verdict}."
)
print(f"Model A Durbin-Watson = {dw_stat:.4f}")
print(dw_interp)
if dw_verdict.startswith("inconclusive"):
    print(
        "The bounds test does not resolve this: DW falls strictly between dL and dU, so neither "
        "'no autocorrelation' nor 'positive autocorrelation' can be concluded from DW alone at 5%."
    )

diag_rows_a.append({
    "order": 1, "test": "Durbin-Watson (Savin-White 5% bounds, n=35, k=6)", "statistic": dw_stat,
    "df": f"dL={DW_DL}, dU={DW_DU}",
    "p_value": np.nan, "decision_at_5pct": dw_verdict, "interpretation": dw_interp,
    "added_beyond_ch3_7": False,
})

Model A Durbin-Watson = 1.5566
DW=1.5566 vs. Savin-White 5% bounds dL=1.16, dU=1.803 (n=35, k=6) -> inconclusive (DW falls in the bounds-test gray zone).
The bounds test does not resolve this: DW falls strictly between dL and dU, so neither 'no autocorrelation' nor 'positive autocorrelation' can be concluded from DW alone at 5%.


In [7]:
BG_LAGS = CAPPED_Q  # model's own lag structure: (p,q) = (1,1) -> primary test at lag 1
bg_lm, bg_lm_p, bg_f, bg_f_p = acorr_breusch_godfrey(model_b_ols_equiv, nlags=BG_LAGS)
bg_interp = (
    f"Breusch-Godfrey LM({BG_LAGS}) = {bg_lm:.4f}, p={bg_lm_p:.4f} -> "
    + ("residual autocorrelation detected at 5%." if bg_lm_p < 0.05 else "no significant residual autocorrelation at 5% (borderline, see lag-2 sensitivity check below).")
)
print(f"Model B Breusch-Godfrey LM test (lag={BG_LAGS}): LM={bg_lm:.4f}, p={bg_lm_p:.4f}")
print(bg_interp)

diag_rows_b.append({
    "order": 1, "test": f"Breusch-Godfrey LM (lag={BG_LAGS}, model's own lag structure)", "statistic": bg_lm, "df": BG_LAGS,
    "p_value": bg_lm_p, "decision_at_5pct": sig_decision(bg_lm_p), "interpretation": bg_interp,
    "added_beyond_ch3_7": False,
})

Model B Breusch-Godfrey LM test (lag=1): LM=3.3192, p=0.0685
Breusch-Godfrey LM(1) = 3.3192, p=0.0685 -> no significant residual autocorrelation at 5% (borderline, see lag-2 sensitivity check below).


### Step 1 sensitivity check -- Breusch-Godfrey at lag 2

Lag-1 came back borderline (p just above 0.05). Re-run at lag 2 to check whether that borderline
read is an artifact of the specific lag order tested, per the model's own `(p,q)=(1,1)` structure
being the leanest possible choice -- not because lag 2 is otherwise implied by the model spec.

In [8]:
BG_LAGS_2 = 2
bg_lm2, bg_lm2_p, bg_f2, bg_f2_p = acorr_breusch_godfrey(model_b_ols_equiv, nlags=BG_LAGS_2)
bg2_sensitivity_note = (
    "CONSISTENT with lag 1: still not significant at 5%." if bg_lm2_p >= 0.05 and bg_lm_p >= 0.05
    else "NOT robust to lag choice: lag 1 was not significant at 5%, but lag 2 IS significant at 5% -- "
         "the borderline lag-1 read understates the case for residual autocorrelation in Model B."
)
bg2_interp = f"Breusch-Godfrey LM({BG_LAGS_2}) = {bg_lm2:.4f}, p={bg_lm2_p:.4f} -> " + (
    "residual autocorrelation detected at 5%." if bg_lm2_p < 0.05 else "no significant residual autocorrelation at 5%."
) + f" {bg2_sensitivity_note}"
print(f"Model B Breusch-Godfrey LM test (lag={BG_LAGS_2}, sensitivity check): LM={bg_lm2:.4f}, p={bg_lm2_p:.4f}")
print(bg2_interp)

diag_rows_b.append({
    "order": "1_sensitivity", "test": f"Breusch-Godfrey LM (lag={BG_LAGS_2}, sensitivity check)", "statistic": bg_lm2, "df": BG_LAGS_2,
    "p_value": bg_lm2_p, "decision_at_5pct": sig_decision(bg_lm2_p), "interpretation": bg2_interp,
    "added_beyond_ch3_7": False,
})

Model B Breusch-Godfrey LM test (lag=2, sensitivity check): LM=6.8412, p=0.0327
Breusch-Godfrey LM(2) = 6.8412, p=0.0327 -> residual autocorrelation detected at 5%. NOT robust to lag choice: lag 1 was not significant at 5%, but lag 2 IS significant at 5% -- the borderline lag-1 read understates the case for residual autocorrelation in Model B.


## Step 2 -- Heteroskedasticity (Breusch-Pagan)

Run on both models. H0 = homoskedastic (constant residual variance).

In [9]:
bp_a_lm, bp_a_p, bp_a_f, bp_a_fp = het_breuschpagan(model_a.resid, model_a.model.exog)
bp_a_df = model_a.model.exog.shape[1] - 1
bp_a_interp = f"Breusch-Pagan LM = {bp_a_lm:.4f} (df={bp_a_df}), p={bp_a_p:.4f} -> " + (
    "heteroskedasticity detected at 5%." if bp_a_p < 0.05 else "no significant heteroskedasticity at 5%."
)
print(f"Model A Breusch-Pagan: LM={bp_a_lm:.4f}, df={bp_a_df}, p={bp_a_p:.4f}")
print(bp_a_interp)

diag_rows_a.append({
    "order": 2, "test": "Breusch-Pagan LM", "statistic": bp_a_lm, "df": bp_a_df,
    "p_value": bp_a_p, "decision_at_5pct": sig_decision(bp_a_p), "interpretation": bp_a_interp,
    "added_beyond_ch3_7": False,
})

Model A Breusch-Pagan: LM=3.6502, df=6, p=0.7239
Breusch-Pagan LM = 3.6502 (df=6), p=0.7239 -> no significant heteroskedasticity at 5%.


In [10]:
bp_b_lm, bp_b_p, bp_b_f, bp_b_fp = het_breuschpagan(model_b_ols_equiv.resid, model_b_ols_equiv.model.exog)
bp_b_df = model_b_ols_equiv.model.exog.shape[1] - 1
bp_b_interp = f"Breusch-Pagan LM = {bp_b_lm:.4f} (df={bp_b_df}), p={bp_b_p:.4f} -> " + (
    "heteroskedasticity detected at 5%." if bp_b_p < 0.05 else "no significant heteroskedasticity at 5%."
)
print(f"Model B Breusch-Pagan: LM={bp_b_lm:.4f}, df={bp_b_df}, p={bp_b_p:.4f}")
print(bp_b_interp)

diag_rows_b.append({
    "order": 2, "test": "Breusch-Pagan LM", "statistic": bp_b_lm, "df": bp_b_df,
    "p_value": bp_b_p, "decision_at_5pct": sig_decision(bp_b_p), "interpretation": bp_b_interp,
    "added_beyond_ch3_7": False,
})

Model B Breusch-Pagan: LM=12.7642, df=13, p=0.4662
Breusch-Pagan LM = 12.7642 (df=13), p=0.4662 -> no significant heteroskedasticity at 5%.


## Step 3 -- Residual normality (Jarque-Bera)

**Flagged explicitly: not in the thesis methodology chapter (Ch. 3.7).** Added here as standard
practice and trivial to reproduce in EViews. H0 = normally distributed residuals. Run on both
models.

In [11]:
jb_a_stat, jb_a_p, jb_a_skew, jb_a_kurt = jarque_bera(model_a.resid)
jb_a_interp = f"JB = {jb_a_stat:.4f} (skew={jb_a_skew:.4f}, kurtosis={jb_a_kurt:.4f}), p={jb_a_p:.4f} -> " + (
    "residuals are not normally distributed at 5%." if jb_a_p < 0.05 else "no evidence against residual normality at 5%."
)
print(f"Model A Jarque-Bera: JB={jb_a_stat:.4f}, p={jb_a_p:.4f}")
print(jb_a_interp)

diag_rows_a.append({
    "order": 3, "test": "Jarque-Bera", "statistic": jb_a_stat, "df": 2,
    "p_value": jb_a_p, "decision_at_5pct": sig_decision(jb_a_p), "interpretation": jb_a_interp,
    "added_beyond_ch3_7": True,
})

Model A Jarque-Bera: JB=0.6192, p=0.7338
JB = 0.6192 (skew=0.2934, kurtosis=2.7167), p=0.7338 -> no evidence against residual normality at 5%.


In [12]:
jb_b_stat, jb_b_p, jb_b_skew, jb_b_kurt = jarque_bera(model_b_ols_equiv.resid)
jb_b_interp = f"JB = {jb_b_stat:.4f} (skew={jb_b_skew:.4f}, kurtosis={jb_b_kurt:.4f}), p={jb_b_p:.4f} -> " + (
    "residuals are not normally distributed at 5%." if jb_b_p < 0.05 else "no evidence against residual normality at 5%."
)
print(f"Model B Jarque-Bera: JB={jb_b_stat:.4f}, p={jb_b_p:.4f}")
print(jb_b_interp)

diag_rows_b.append({
    "order": 3, "test": "Jarque-Bera", "statistic": jb_b_stat, "df": 2,
    "p_value": jb_b_p, "decision_at_5pct": sig_decision(jb_b_p), "interpretation": jb_b_interp,
    "added_beyond_ch3_7": True,
})

Model B Jarque-Bera: JB=0.3372, p=0.8449
JB = 0.3372 (skew=-0.2439, kurtosis=2.9895), p=0.8449 -> no evidence against residual normality at 5%.


## Step 4 -- Functional form (Ramsey RESET)

**Flagged explicitly: not in the thesis methodology chapter (Ch. 3.7).** Added for the same
reason as Jarque-Bera above. Augments the regression with powers of the fitted values
(`fitted^2`, `fitted^3`) and tests their joint significance via an F-test. H0 = no misspecification.
Run on both models.

In [13]:
reset_a = linear_reset(model_a, power=3, test_type="fitted", use_f=True)
reset_a_interp = f"RESET F({reset_a.df_num:.0f},{reset_a.df_denom:.0f}) = {reset_a.fvalue:.4f}, p={reset_a.pvalue:.4f} -> " + (
    "functional-form misspecification detected at 5%." if reset_a.pvalue < 0.05 else "no evidence of functional-form misspecification at 5%."
)
print(f"Model A Ramsey RESET: F={reset_a.fvalue:.4f}, p={reset_a.pvalue:.4f}")
print(reset_a_interp)

diag_rows_a.append({
    "order": 4, "test": "Ramsey RESET", "statistic": reset_a.fvalue, "df": f"({reset_a.df_num:.0f},{reset_a.df_denom:.0f})",
    "p_value": reset_a.pvalue, "decision_at_5pct": sig_decision(reset_a.pvalue), "interpretation": reset_a_interp,
    "added_beyond_ch3_7": True,
})

Model A Ramsey RESET: F=0.0112, p=0.9888
RESET F(2,26) = 0.0112, p=0.9888 -> no evidence of functional-form misspecification at 5%.


In [14]:
reset_b = linear_reset(model_b_ols_equiv, power=3, test_type="fitted", use_f=True)
reset_b_interp = f"RESET F({reset_b.df_num:.0f},{reset_b.df_denom:.0f}) = {reset_b.fvalue:.4f}, p={reset_b.pvalue:.4f} -> " + (
    "functional-form misspecification detected at 5%." if reset_b.pvalue < 0.05 else "no evidence of functional-form misspecification at 5%."
)
print(f"Model B Ramsey RESET: F={reset_b.fvalue:.4f}, p={reset_b.pvalue:.4f}")
print(reset_b_interp)

diag_rows_b.append({
    "order": 4, "test": "Ramsey RESET", "statistic": reset_b.fvalue, "df": f"({reset_b.df_num:.0f},{reset_b.df_denom:.0f})",
    "p_value": reset_b.pvalue, "decision_at_5pct": sig_decision(reset_b.pvalue), "interpretation": reset_b_interp,
    "added_beyond_ch3_7": True,
})

Model B Ramsey RESET: F=1.6822, p=0.2139
RESET F(2,18) = 1.6822, p=0.2139 -> no evidence of functional-form misspecification at 5%.


## Diagnostics for the first-differenced OLS

Diagnosed here per the plan's original Branch-B scope (run diagnostics on the first-differenced
OLS alongside Model A/B). Its results below are what later motivate redesignating it as the
**primary model for H1/H2 inference** in Step 6c -- at the time these diagnostics are run, it is
still being evaluated as a comparison model; the redesignation decision comes after.

Per this plan's "Decisions & flags" section, diagnostics run on the first-differenced OLS too,
since Branch B applies. It has no lagged dependent variable, so -- like Model A -- Durbin-Watson
is the correct autocorrelation test (not Breusch-Godfrey). Same four tests, same order, as Model A.

In [15]:
# Savin & White (1977) 5% bounds, n=34 (one observation lost to differencing), k=6 regressors.
DIFF_N, DIFF_K = int(model_diff.nobs), diff_X.shape[1]
assert (DIFF_N, DIFF_K) == (34, 6), (
    f"Savin-White dL/dU below are hardcoded for n=34, k=6; got n={DIFF_N}, k={DIFF_K} -- look up "
    "the correct table row before proceeding."
)
DIFF_DW_DL, DIFF_DW_DU = 1.144, 1.807

dw_diff_stat = durbin_watson(model_diff.resid)
dw_diff_verdict = dw_bounds_verdict(dw_diff_stat, DIFF_DW_DL, DIFF_DW_DU)
dw_diff_interp = (
    f"DW={dw_diff_stat:.4f} vs. Savin-White 5% bounds dL={DIFF_DW_DL}, dU={DIFF_DW_DU} (n=34, k=6) -> {dw_diff_verdict}."
)
print(f"First-differenced OLS Durbin-Watson = {dw_diff_stat:.4f}")
print(dw_diff_interp)
if dw_diff_verdict.startswith("inconclusive"):
    print("Same as Model A: the bounds test does not resolve this at 5%.")

diag_rows_diff.append({
    "order": 1, "test": "Durbin-Watson (Savin-White 5% bounds, n=34, k=6)", "statistic": dw_diff_stat,
    "df": f"dL={DIFF_DW_DL}, dU={DIFF_DW_DU}",
    "p_value": np.nan, "decision_at_5pct": dw_diff_verdict, "interpretation": dw_diff_interp,
    "added_beyond_ch3_7": False,
})

First-differenced OLS Durbin-Watson = 2.3812
DW=2.3812 vs. Savin-White 5% bounds dL=1.144, dU=1.807 (n=34, k=6) -> inconclusive (DW falls in the bounds-test gray zone).
Same as Model A: the bounds test does not resolve this at 5%.


In [16]:
bp_diff_lm, bp_diff_p, bp_diff_f, bp_diff_fp = het_breuschpagan(model_diff.resid, model_diff.model.exog)
bp_diff_df = model_diff.model.exog.shape[1] - 1
bp_diff_interp = f"Breusch-Pagan LM = {bp_diff_lm:.4f} (df={bp_diff_df}), p={bp_diff_p:.4f} -> " + (
    "heteroskedasticity detected at 5%." if bp_diff_p < 0.05 else "no significant heteroskedasticity at 5%."
)
print(f"First-differenced OLS Breusch-Pagan: LM={bp_diff_lm:.4f}, df={bp_diff_df}, p={bp_diff_p:.4f}")
print(bp_diff_interp)

diag_rows_diff.append({
    "order": 2, "test": "Breusch-Pagan LM", "statistic": bp_diff_lm, "df": bp_diff_df,
    "p_value": bp_diff_p, "decision_at_5pct": sig_decision(bp_diff_p), "interpretation": bp_diff_interp,
    "added_beyond_ch3_7": False,
})

First-differenced OLS Breusch-Pagan: LM=2.1496, df=6, p=0.9054
Breusch-Pagan LM = 2.1496 (df=6), p=0.9054 -> no significant heteroskedasticity at 5%.


In [17]:
jb_diff_stat, jb_diff_p, jb_diff_skew, jb_diff_kurt = jarque_bera(model_diff.resid)
jb_diff_interp = f"JB = {jb_diff_stat:.4f} (skew={jb_diff_skew:.4f}, kurtosis={jb_diff_kurt:.4f}), p={jb_diff_p:.4f} -> " + (
    "residuals are not normally distributed at 5%." if jb_diff_p < 0.05 else "no evidence against residual normality at 5%."
)
print(f"First-differenced OLS Jarque-Bera: JB={jb_diff_stat:.4f}, p={jb_diff_p:.4f}")
print(jb_diff_interp)

diag_rows_diff.append({
    "order": 3, "test": "Jarque-Bera", "statistic": jb_diff_stat, "df": 2,
    "p_value": jb_diff_p, "decision_at_5pct": sig_decision(jb_diff_p), "interpretation": jb_diff_interp,
    "added_beyond_ch3_7": True,
})

First-differenced OLS Jarque-Bera: JB=0.2568, p=0.8795
JB = 0.2568 (skew=-0.1701, kurtosis=3.2559), p=0.8795 -> no evidence against residual normality at 5%.


In [18]:
reset_diff = linear_reset(model_diff, power=3, test_type="fitted", use_f=True)
reset_diff_interp = f"RESET F({reset_diff.df_num:.0f},{reset_diff.df_denom:.0f}) = {reset_diff.fvalue:.4f}, p={reset_diff.pvalue:.4f} -> " + (
    "functional-form misspecification detected at 5%." if reset_diff.pvalue < 0.05 else "no evidence of functional-form misspecification at 5%."
)
print(f"First-differenced OLS Ramsey RESET: F={reset_diff.fvalue:.4f}, p={reset_diff.pvalue:.4f}")
print(reset_diff_interp)

diag_rows_diff.append({
    "order": 4, "test": "Ramsey RESET", "statistic": reset_diff.fvalue, "df": f"({reset_diff.df_num:.0f},{reset_diff.df_denom:.0f})",
    "p_value": reset_diff.pvalue, "decision_at_5pct": sig_decision(reset_diff.pvalue), "interpretation": reset_diff_interp,
    "added_beyond_ch3_7": True,
})

First-differenced OLS Ramsey RESET: F=1.6357, p=0.2150
RESET F(2,25) = 1.6357, p=0.2150 -> no evidence of functional-form misspecification at 5%.


In [19]:
diagnostics_model_diff = pd.DataFrame(diag_rows_diff)
diagnostics_model_diff.to_csv(DIAG_DIFF_OUT, index=False)
print(f"Written -> {DIAG_DIFF_OUT}")
diagnostics_model_diff

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/diagnostics_model_diff_ols.csv


,order,test,statistic,df,p_value,decision_at_5pct,interpretation,added_beyond_ch3_7
0,1,"Durbin-Watson (Savin-White 5% bounds, n=34, k=6)",2.381219,"dL=1.144, dU=1.807",NaN,inconclusive (DW falls in the bounds-test gray...,"DW=2.3812 vs. Savin-White 5% bounds dL=1.144, ...",False
1,2,Breusch-Pagan LM,2.149587,6,0.905440,fail to reject H0 (p >= 0.05),"Breusch-Pagan LM = 2.1496 (df=6), p=0.9054 -> ...",False
2,3,Jarque-Bera,0.256820,2,0.879493,fail to reject H0 (p >= 0.05),"JB = 0.2568 (skew=-0.1701, kurtosis=3.2559), p...",True
3,4,Ramsey RESET,1.635665,"(2,25)",0.214992,fail to reject H0 (p >= 0.05),"RESET F(2,25) = 1.6357, p=0.2150 -> no evidenc...",True


## Step 5 -- ARDL-specific diagnostics (Model B only)

### Step 5a -- Bounds-test F-statistic, restated

Already computed in step vi (Step B8) -- restated here, not recomputed, for a complete
single-place diagnostics summary, since conceptually it is a pre-condition check before the
long-run coefficients can be trusted. Case 3 (unrestricted intercept, no trend), k=6 regressors.

In [20]:
capped_bounds = pd.read_csv(CAPPED_BOUNDS_IN)
row_5pct = capped_bounds.loc[capped_bounds["confidence_level_pct"] == 95.0].iloc[0]

if row_5pct["above_upper"]:
    bounds_verdict = "cointegration CONFIRMED (F-stat above upper bound at 5%)"
elif row_5pct["below_lower"]:
    bounds_verdict = "NO cointegration (F-stat below lower bound at 5%)"
else:
    bounds_verdict = "INCONCLUSIVE at 5% (F-stat between bounds) -- restated from step vi, not re-derived here"

bounds_interp = (
    f"Bounds F-stat = {row_5pct['f_stat']:.4f} vs. 5% bounds [{row_5pct['crit_lower']:.3f}, {row_5pct['crit_upper']:.3f}] "
    f"(Case {int(row_5pct['case'])}, k={int(row_5pct['k_regressors'])}) -> {bounds_verdict}"
)
print(bounds_interp)

diag_rows_b.append({
    "order": "5a", "test": "ARDL bounds F-test (restated from step vi)", "statistic": row_5pct["f_stat"],
    "df": f"k={int(row_5pct['k_regressors'])}, Case {int(row_5pct['case'])}",
    "p_value": np.nan, "decision_at_5pct": bounds_verdict, "interpretation": bounds_interp,
    "added_beyond_ch3_7": False,
})

Bounds F-stat = 2.3658 vs. 5% bounds [2.328, 3.500] (Case 3, k=6) -> INCONCLUSIVE at 5% (F-stat between bounds) -- restated from step vi, not re-derived here


### Step 5b -- CUSUM and CUSUMSQ stability tests

Both require the classic Brown, Durbin & Evans (1975) **recursive** residuals -- re-estimating
the model observation-by-observation from an initial window of `k` observations (`k` = number of
parameters, 14 for the capped ARDL(1,1)) onward. Attempted below at the standard `skip=k`, with
the failure reason inspected before deciding whether to force a larger, non-standard skip.

In [21]:
K_PARAMS = len(model_b_ols_equiv.params)
N_OBS_B = int(model_b_ols_equiv.nobs)

cusum_feasible = False
cusum_reason = ""
try:
    recursive_olsresiduals(model_b_ols_equiv, skip=K_PARAMS)
    cusum_feasible = True
except Exception as exc:
    cusum_reason = str(exc).strip()

shock = frame["shock"].values
first_shock_idx = int(np.argmax(shock == 1))  # first index where SHOCK == 1

print(f"n_obs={N_OBS_B}, k_params={K_PARAMS} -> standard skip=k leaves at most {N_OBS_B - K_PARAMS} recursive residuals even if feasible.")
print(f"SHOCK is 0 for the first {first_shock_idx} of {len(shock)} annual observations (first 1990-{frame.index[first_shock_idx - 1]}, first shock year {frame.index[first_shock_idx]}).")
print(f"recursive_olsresiduals(skip={K_PARAMS}) feasible: {cusum_feasible}")
if not cusum_feasible:
    print(f"Failure: {cusum_reason}")

n_obs=34, k_params=14 -> standard skip=k leaves at most 20 recursive residuals even if feasible.
SHOCK is 0 for the first 18 of 35 annual observations (first 1990-2007, first shock year 2008).
recursive_olsresiduals(skip=14) feasible: False
Failure: "The initial regressor matrix, x[:skip], issingular. You must use a value of
skip large enough to ensure that the first OLS estimator is well-defined.


In [22]:
if not cusum_feasible:
    # Diagnose exactly how large `skip` would need to be for a non-singular initial window,
    # to state precisely how thin the resulting recursive-residual path would be.
    min_feasible_skip = None
    for skip in range(K_PARAMS, N_OBS_B):
        try:
            recursive_olsresiduals(model_b_ols_equiv, skip=skip)
            min_feasible_skip = skip
            break
        except Exception:
            continue

    if min_feasible_skip is None:
        cusum_status = "skipped -- infeasible at any skip within the sample"
        cusum_skip_detail = "no skip value in range produced a non-singular initial window"
        n_recursive_at_min_skip = np.nan
    else:
        n_recursive_at_min_skip = N_OBS_B - min_feasible_skip
        cusum_status = "skipped -- infeasible at the standard skip=k; not forced at a non-standard skip"
        cusum_skip_detail = (
            f"smallest non-singular skip found = {min_feasible_skip} (vs. standard skip=k={K_PARAMS}), "
            f"leaving only {n_recursive_at_min_skip} recursive residuals out of {N_OBS_B} observations"
        )
    print(cusum_skip_detail)
else:
    cusum_status = "computed"
    cusum_skip_detail = ""
    min_feasible_skip = K_PARAMS
    n_recursive_at_min_skip = N_OBS_B - K_PARAMS

cusum_reason_full = (
    "The SHOCK dummy is 0 for the first 18 of 34 usable observations (financial-crisis years only "
    "start in 2008), so any initial regressor window at the standard skip=k=14 has a zero SHOCK "
    "column and is rank-deficient (singular X'X). " + cusum_skip_detail + ". Per the plan's explicit "
    "allowance, this is a data-driven infeasibility given the sample size, not a silent omission: "
    "CUSUM and CUSUMSQ are both skipped for Model B rather than forcing an unreliable result from a "
    "non-standard skip or a near-empty recursive-residual path."
) if not cusum_feasible else "Recursive residuals were feasible at the standard skip=k; see the CUSUM/CUSUMSQ table below."

print(cusum_reason_full)

cusum_status_table = pd.DataFrame([{
    "test": "CUSUM", "status": cusum_status, "n_obs": N_OBS_B, "k_params": K_PARAMS,
    "standard_skip": K_PARAMS, "min_feasible_skip": min_feasible_skip,
    "n_recursive_residuals_if_forced": n_recursive_at_min_skip, "reason": cusum_reason_full,
}, {
    "test": "CUSUMSQ", "status": cusum_status, "n_obs": N_OBS_B, "k_params": K_PARAMS,
    "standard_skip": K_PARAMS, "min_feasible_skip": min_feasible_skip,
    "n_recursive_residuals_if_forced": n_recursive_at_min_skip, "reason": cusum_reason_full,
}])
cusum_status_table.to_csv(CUSUM_STATUS_OUT, index=False)
print(f"Written -> {CUSUM_STATUS_OUT}")

diag_rows_b.append({
    "order": "5b", "test": "CUSUM / CUSUMSQ stability (Brown-Durbin-Evans)", "statistic": np.nan, "df": np.nan,
    "p_value": np.nan, "decision_at_5pct": cusum_status, "interpretation": cusum_reason_full,
    "added_beyond_ch3_7": False,
})
cusum_status_table

smallest non-singular skip found = 19 (vs. standard skip=k=14), leaving only 15 recursive residuals out of 34 observations
The SHOCK dummy is 0 for the first 18 of 34 usable observations (financial-crisis years only start in 2008), so any initial regressor window at the standard skip=k=14 has a zero SHOCK column and is rank-deficient (singular X'X). smallest non-singular skip found = 19 (vs. standard skip=k=14), leaving only 15 recursive residuals out of 34 observations. Per the plan's explicit allowance, this is a data-driven infeasibility given the sample size, not a silent omission: CUSUM and CUSUMSQ are both skipped for Model B rather than forcing an unreliable result from a non-standard skip or a near-empty recursive-residual path.
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/cusum_cusumsq_model_b_status.csv


,test,status,n_obs,k_params,standard_skip,min_feasible_skip,n_recursive_residuals_if_forced,reason
0,CUSUM,skipped -- infeasible at the standard skip=k; ...,34,14,14,19,15,The SHOCK dummy is 0 for the first 18 of 34 us...
1,CUSUMSQ,skipped -- infeasible at the standard skip=k; ...,34,14,14,19,15,The SHOCK dummy is 0 for the first 18 of 34 us...


## Step 6 -- Heteroskedasticity/autocorrelation remediation

Per the plan, remediation is scoped to the OLS model (Model A). The plan's literal trigger is "if
Step 1 and/or Step 2 detects a problem" -- Step 2 (Breusch-Pagan) is clean, but Step 1's Savin-White
DW bounds test came back **inconclusive**, not clean, at 5%. Rather than treat "inconclusive" as
equivalent to "no problem," Newey-West HAC standard errors are computed and reported side by side
with the original OLS SEs unconditionally, as a precautionary robustness check and a cleaner
EViews cross-check -- protecting against both autocorrelation and heteroskedasticity regardless of
whether a formal rejection strictly triggered it. `maxlags` uses the common default
`floor(4*(T/100)^(2/9))`.

**Step 6b below extends the same HAC-robust treatment to Model B's long-run and short-run
coefficients** -- outside the plan's literal scope, but requested explicitly given Step 1's
lag-2 Breusch-Godfrey result (significant at 5%), which is exactly the kind of residual
autocorrelation Newey-West is designed to protect against.

In [23]:
NEWEY_WEST_MAXLAGS = int(np.floor(4 * (model_a.nobs / 100) ** (2 / 9)))
print(f"Newey-West maxlags: floor(4*({int(model_a.nobs)}/100)^(2/9)) = {NEWEY_WEST_MAXLAGS}")

hac_formally_triggered = bool(dw_verdict == "positive autocorrelation (DW < dL)" or (bp_a_p < 0.05))
print(
    f"Formally triggered by a clean rejection at 5%: {hac_formally_triggered} "
    f"(Model A DW verdict='{dw_verdict}', Breusch-Pagan p={bp_a_p:.4f}). "
    "Computed regardless, per the DW bounds test being inconclusive rather than clean."
)

Newey-West maxlags: floor(4*(35/100)^(2/9)) = 3
Formally triggered by a clean rejection at 5%: False (Model A DW verdict='inconclusive (DW falls in the bounds-test gray zone)', Breusch-Pagan p=0.7239). Computed regardless, per the DW bounds test being inconclusive rather than clean.


In [24]:
model_a_hac = model_a.get_robustcov_results(cov_type="HAC", maxlags=NEWEY_WEST_MAXLAGS)
hac_comparison = pd.DataFrame({
    "term": model_a.params.index,
    "coef": model_a.params.values,
    "std_err_original": model_a.bse.values,
    "t_stat_original": model_a.tvalues.values,
    "p_value_original": model_a.pvalues.values,
    "std_err_hac": model_a_hac.bse,
    "t_stat_hac": model_a_hac.tvalues,
    "p_value_hac": model_a_hac.pvalues,
})
hac_comparison.to_csv(HAC_COEF_OUT, index=False)
print(f"Written -> {HAC_COEF_OUT}")

hac_status = pd.DataFrame([{
    "computed": True,
    "formally_triggered_by_clean_rejection": hac_formally_triggered,
    "reason": (
        f"Computed regardless of a strict trigger: Savin-White DW bounds test on Model A came back "
        f"inconclusive at 5% (DW={dw_stat:.4f}, dL={DW_DL}, dU={DW_DU}), and Breusch-Pagan was clean "
        f"(p={bp_a_p:.4f}). Reported as a precautionary robustness check and EViews cross-check, not "
        "because a formal rejection strictly triggered it."
    ),
    "maxlags": NEWEY_WEST_MAXLAGS, "maxlags_formula": "floor(4*(T/100)^(2/9))", "T": int(model_a.nobs),
    "dw_bounds_test_verdict": dw_verdict, "breusch_pagan_p_value": bp_a_p,
}])
hac_status.to_csv(HAC_STATUS_OUT, index=False)
print(f"Written -> {HAC_STATUS_OUT}")
hac_comparison

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_a_hac_coefficients.csv
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/hac_remediation_status.csv


,term,coef,std_err_original,t_stat_original,p_value_original,std_err_hac,t_stat_hac,p_value_hac
0,const,-2.334482,0.730522,-3.195634,0.003443,0.737064,-3.167273,0.003699
1,DIVP,1.268049,0.602285,2.105398,0.044354,0.573959,2.209304,0.035503
2,DIVM,-0.805481,0.887658,-0.907423,0.371927,0.636277,-1.265928,0.215971
3,INF,-0.004062,0.002733,-1.485997,0.148454,0.002989,-1.359012,0.184990
4,EXR,-0.000941,0.000408,-2.307866,0.028610,0.000467,-2.016657,0.053414
5,log(FDI),0.142768,0.031814,4.487613,0.000112,0.032401,4.406243,0.000140
6,SHOCK,-0.020333,0.068522,-0.296735,0.768859,0.044523,-0.456678,0.651426


In [25]:
# Explicit size-of-shift check for DIVP, called out by name since it is one of the H1 regressors:
# report the exact original (non-HAC) p-value alongside the HAC p-value, rather than letting the
# HAC number stand alone.
divp_row = hac_comparison.loc[hac_comparison["term"] == "DIVP"].iloc[0]
divp_shift_note = (
    f"DIVP: original (non-HAC) p-value = {divp_row['p_value_original']:.4f} "
    f"(already significant at 5% under the original OLS SEs), HAC p-value = {divp_row['p_value_hac']:.4f}. "
    "HAC tightens the SE and the p-value moves slightly, but DIVP does not cross a significance "
    "threshold here -- it was significant before HAC and remains significant after."
)
print(divp_shift_note)

DIVP: original (non-HAC) p-value = 0.0444 (already significant at 5% under the original OLS SEs), HAC p-value = 0.0355. HAC tightens the SE and the p-value moves slightly, but DIVP does not cross a significance threshold here -- it was significant before HAC and remains significant after.


## Step 6b -- HAC-robust standard errors for Model B (long-run and short-run coefficients)

Requested explicitly given Step 1's lag-2 Breusch-Godfrey result (LM(2) significant at 5%,
p=0.0327) -- Model B's own reported SEs come from the same non-robust OLS-based UECM estimation as
Model A's original SEs, so the same residual-autocorrelation exposure applies. `maxlags` is
recomputed for Model B's own sample size (T=34, one fewer than Model A's T=35 due to the ARDL lag).

**Caveat, stated explicitly:** Model B includes a lagged dependent variable (`ERI.L1`) as a
regressor by construction (the error-correction term). Newey-West HAC standard errors remain a
standard, widely-used robustness check in this setting (as in Model A), but the textbook HAC
asymptotics were derived for strictly exogenous regressors; with a lagged dependent variable the
same practical caveats that apply to any dynamic-model HAC application apply here too. Reported as
a robustness cross-check for that reason, not a replacement for the original SEs.

**Long-run coefficients (`β_LR,k = θ_k / θ1`):** the original SEs use the delta method on the
non-robust covariance matrix (`UECMResults.ci_bse`, reproduced manually below and verified to
match step vi's saved values exactly). The HAC-robust long-run SEs apply the identical delta-method
Jacobian to the HAC covariance matrix instead -- same transformation, different covariance input.

**Short-run coefficients (the `D.`-prefixed terms):** these are direct linear coefficients in the
OLS-equivalent design (no delta-method transform needed), so their HAC SEs come straight from the
HAC-robust covariance matrix's diagonal, exactly as for Model A.

In [26]:
MAXLAGS_B = int(np.floor(4 * (model_b_ols_equiv.nobs / 100) ** (2 / 9)))
print(f"Model B Newey-West maxlags: floor(4*({int(model_b_ols_equiv.nobs)}/100)^(2/9)) = {MAXLAGS_B}")

model_b_hac = model_b_ols_equiv.get_robustcov_results(cov_type="HAC", maxlags=MAXLAGS_B)
_param_names_b = list(model_b_ols_equiv.params.index)
cov_orig_b = pd.DataFrame(model_b_ols_equiv.cov_params(), index=_param_names_b, columns=_param_names_b)
cov_hac_b = pd.DataFrame(model_b_hac.cov_params(), index=_param_names_b, columns=_param_names_b)
bse_hac_b = pd.Series(model_b_hac.bse, index=_param_names_b)

THETA1_NAME = "ERI.L1"
theta1_b = model_b_ols_equiv.params[THETA1_NAME]
level_terms_b = ["const"] + [c for c in _param_names_b if c.endswith(".L1") and c != THETA1_NAME]


def delta_lr_se(theta_k_name, cov):
    theta_k = model_b_ols_equiv.params[theta_k_name]
    dgk = 1 / theta1_b
    dg1 = -theta_k / theta1_b ** 2
    var = (
        dgk ** 2 * cov.loc[theta_k_name, theta_k_name]
        + dg1 ** 2 * cov.loc[THETA1_NAME, THETA1_NAME]
        + 2 * dgk * dg1 * cov.loc[theta_k_name, THETA1_NAME]
    )
    return np.sqrt(var)


saved_long_run = pd.read_csv(OUTPUT_DIR / "ardl_capped_1_1_long_run_coefficients.csv").set_index("term")
df_resid_b = int(model_b_ols_equiv.df_resid)

lr_rows = []
for term in level_terms_b:
    coef = model_b_ols_equiv.params[term] / theta1_b
    se_orig_manual = delta_lr_se(term, cov_orig_b)
    se_hac = delta_lr_se(term, cov_hac_b)
    t_hac = coef / se_hac
    p_hac = 2 * sstats.t.sf(abs(t_hac), df_resid_b)
    lr_rows.append({
        "term": term, "long_run_coef": coef,
        "std_err_original": se_orig_manual, "std_err_hac": se_hac,
        "t_stat_hac": t_hac, "p_value_hac": p_hac,
    })

hac_long_run = pd.DataFrame(lr_rows)

# Sanity check: the manually-reconstructed original SEs must match step vi's saved delta-method
# SEs exactly (same non-robust covariance, same Jacobian) before the HAC column is trusted.
lr_check = hac_long_run.set_index("term")["std_err_original"]
lr_saved = saved_long_run["std_err_delta_method"]
lr_max_diff = (lr_check.reindex(lr_saved.index) - lr_saved).abs().max()
assert lr_max_diff < 1e-6, f"Manual delta-method SE does not match step vi's saved long-run SEs (max abs diff={lr_max_diff})"
print(f"Delta-method reconstruction verified against step vi's saved long-run SEs (max abs diff={lr_max_diff:.2e})")

hac_long_run.to_csv(HAC_B_LONG_RUN_OUT, index=False)
print(f"Written -> {HAC_B_LONG_RUN_OUT}")
hac_long_run

Model B Newey-West maxlags: floor(4*(34/100)^(2/9)) = 3
Delta-method reconstruction verified against step vi's saved long-run SEs (max abs diff=0.00e+00)
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_long_run_hac_comparison.csv


,term,long_run_coef,std_err_original,std_err_hac,t_stat_hac,p_value_hac
0,const,1.677979,1.329219,0.846616,1.981983,0.061392
1,DIVP.L1,-1.045092,0.982920,0.676220,-1.545493,0.137904
2,DIVM.L1,1.205365,1.679532,0.691504,1.743105,0.096670
3,INF.L1,0.002154,0.007343,0.006026,0.357480,0.724477
4,EXR.L1,0.000055,0.000890,0.000663,0.083195,0.934523
5,log(FDI).L1,-0.130140,0.059393,0.038429,-3.386475,0.002931
6,SHOCK.L1,0.022792,0.130576,0.071540,0.318582,0.753346


In [27]:
saved_short_run = pd.read_csv(OUTPUT_DIR / "ardl_capped_1_1_short_run_coefficients.csv").set_index("term")
short_terms_b = [c for c in _param_names_b if c.startswith("D.")]

sr_check = model_b_ols_equiv.bse[short_terms_b]
sr_max_diff = (sr_check.reindex(saved_short_run.index) - saved_short_run["std_err"]).abs().max()
assert sr_max_diff < 1e-6, f"OLS-equivalent short-run SEs do not match step vi's saved short-run SEs (max abs diff={sr_max_diff})"
print(f"OLS-equivalent short-run SEs verified against step vi's saved values (max abs diff={sr_max_diff:.2e})")

hac_short_run = pd.DataFrame({
    "term": short_terms_b,
    "coef": model_b_ols_equiv.params[short_terms_b].values,
    "std_err_original": model_b_ols_equiv.bse[short_terms_b].values,
    "std_err_hac": bse_hac_b[short_terms_b].values,
})
hac_short_run["t_stat_hac"] = hac_short_run["coef"] / hac_short_run["std_err_hac"]
hac_short_run["p_value_hac"] = 2 * sstats.t.sf(hac_short_run["t_stat_hac"].abs(), df_resid_b)
hac_short_run.to_csv(HAC_B_SHORT_RUN_OUT, index=False)
print(f"Written -> {HAC_B_SHORT_RUN_OUT}")
hac_short_run

OLS-equivalent short-run SEs verified against step vi's saved values (max abs diff=8.33e-17)
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_short_run_hac_comparison.csv


,term,coef,std_err_original,std_err_hac,t_stat_hac,p_value_hac
0,D.DIVP.L0,1.374798,1.610284,1.670046,0.823210,0.420097
1,D.DIVM.L0,-0.948627,1.169724,0.888067,-1.068192,0.298158
2,D.INF.L0,0.003660,0.006638,0.005320,0.687936,0.499395
3,D.EXR.L0,-0.004547,0.002730,0.002339,-1.944251,0.066067
4,D.log(FDI).L0,0.115607,0.058387,0.039559,2.922380,0.008422
5,D.SHOCK.L0,-0.012942,0.086264,0.082269,-0.157311,0.876577


## Step 6c -- HAC-robust standard errors for the first-differenced OLS

**The first-differenced OLS was redesignated as the primary model for H1/H2 inference** (see
`outputs/modeling_path_decision.csv` addendum and `research_plan.md`'s 2026-07-17 Update note),
based on the ARDL bounds test not confirming cointegration and Model B's lag-2 Breusch-Godfrey
result. Since it is now the primary model, and its own Durbin-Watson result landed in the
**inconclusive** zone (see its Durbin-Watson result above) rather than a clean pass -- the same situation that
triggered HAC for Model A -- the identical precautionary treatment is applied here: Newey-West
HAC-robust SEs computed and reported side by side with the original SEs, unconditionally.
`maxlags` uses the same formula, evaluated at this model's own T=34.

DIVP and DIVM are surfaced explicitly below, since they are the coefficients H1/H2 are tested
against and DIVP's baseline result is only marginal (10% level, not 5%).

In [28]:
MAXLAGS_DIFF = int(np.floor(4 * (model_diff.nobs / 100) ** (2 / 9)))
print(f"First-differenced OLS Newey-West maxlags: floor(4*({int(model_diff.nobs)}/100)^(2/9)) = {MAXLAGS_DIFF}")

model_diff_hac = model_diff.get_robustcov_results(cov_type="HAC", maxlags=MAXLAGS_DIFF)
hac_diff_comparison = pd.DataFrame({
    "term": model_diff.params.index,
    "coef": model_diff.params.values,
    "std_err_original": model_diff.bse.values,
    "t_stat_original": model_diff.tvalues.values,
    "p_value_original": model_diff.pvalues.values,
    "std_err_hac": model_diff_hac.bse,
    "t_stat_hac": model_diff_hac.tvalues,
    "p_value_hac": model_diff_hac.pvalues,
})
hac_diff_comparison.to_csv(MODEL_DIFF_HAC_OUT, index=False)
print(f"Written -> {MODEL_DIFF_HAC_OUT}")
hac_diff_comparison

First-differenced OLS Newey-West maxlags: floor(4*(34/100)^(2/9)) = 3
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_diff_hac_coefficients.csv


,term,coef,std_err_original,t_stat_original,p_value_original,std_err_hac,t_stat_hac,p_value_hac
0,const,0.030579,0.029903,1.022631,0.315558,0.018307,1.670361,0.106402
1,DIVP,2.352928,1.379395,1.705768,0.099531,1.184835,1.985869,0.057288
2,DIVM,-1.402368,1.043964,-1.343310,0.190354,0.801537,-1.749599,0.091547
3,INF,0.000152,0.003758,0.040399,0.968073,0.002605,0.058274,0.953960
4,EXR,-0.003543,0.001800,-1.968141,0.059399,0.000972,-3.646152,0.001120
5,log(FDI),0.117277,0.053479,2.192957,0.037104,0.044665,2.625710,0.014068
6,SHOCK,-0.010492,0.086296,-0.121586,0.904128,0.069049,-0.151956,0.880352


In [29]:
for term in ["DIVP", "DIVM"]:
    row = hac_diff_comparison.loc[hac_diff_comparison["term"] == term].iloc[0]
    orig_sig = "significant at 10%" if row["p_value_original"] < 0.10 else "not significant at 10%"
    hac_sig = "significant at 10%" if row["p_value_hac"] < 0.10 else "not significant at 10%"
    shift = (
        "STRENGTHENS" if row["p_value_hac"] < row["p_value_original"] - 1e-9
        else "WEAKENS" if row["p_value_hac"] > row["p_value_original"] + 1e-9
        else "UNCHANGED"
    )
    print(
        f"{term}: coef={row['coef']:.4f}, original p={row['p_value_original']:.4f} ({orig_sig}), "
        f"HAC p={row['p_value_hac']:.4f} ({hac_sig}) -> {shift} under HAC."
    )

DIVP: coef=2.3529, original p=0.0995 (significant at 10%), HAC p=0.0573 (significant at 10%) -> STRENGTHENS under HAC.
DIVM: coef=-1.4024, original p=0.1904 (not significant at 10%), HAC p=0.0915 (significant at 10%) -> STRENGTHENS under HAC.


## Assemble Model A / Model B diagnostics tables

(The first-differenced OLS table, `diagnostics_model_diff_ols.csv`, was already assembled and
written in its own section above.)

In [30]:
diagnostics_model_a = pd.DataFrame(diag_rows_a)
diagnostics_model_a.to_csv(DIAG_A_OUT, index=False)
print(f"Written -> {DIAG_A_OUT}")
diagnostics_model_a

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/diagnostics_model_a_ols.csv


,order,test,statistic,df,p_value,decision_at_5pct,interpretation,added_beyond_ch3_7
0,1,"Durbin-Watson (Savin-White 5% bounds, n=35, k=6)",1.556572,"dL=1.16, dU=1.803",NaN,inconclusive (DW falls in the bounds-test gray...,"DW=1.5566 vs. Savin-White 5% bounds dL=1.16, d...",False
1,2,Breusch-Pagan LM,3.650234,6,0.723886,fail to reject H0 (p >= 0.05),"Breusch-Pagan LM = 3.6502 (df=6), p=0.7239 -> ...",False
2,3,Jarque-Bera,0.619162,2,0.733754,fail to reject H0 (p >= 0.05),"JB = 0.6192 (skew=0.2934, kurtosis=2.7167), p=...",True
3,4,Ramsey RESET,0.011244,"(2,26)",0.988824,fail to reject H0 (p >= 0.05),"RESET F(2,26) = 0.0112, p=0.9888 -> no evidenc...",True


In [31]:
diagnostics_model_b = pd.DataFrame(diag_rows_b)
diagnostics_model_b.to_csv(DIAG_B_OUT, index=False)
print(f"Written -> {DIAG_B_OUT}")
diagnostics_model_b

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/diagnostics_model_b_ardl_capped.csv


,order,test,statistic,df,p_value,decision_at_5pct,interpretation,added_beyond_ch3_7
0,1,"Breusch-Godfrey LM (lag=1, model's own lag str...",3.319242,1,0.068473,fail to reject H0 (p >= 0.05),"Breusch-Godfrey LM(1) = 3.3192, p=0.0685 -> no...",False
1,1_sensitivity,"Breusch-Godfrey LM (lag=2, sensitivity check)",6.841202,2,0.032693,reject H0 (p < 0.05),"Breusch-Godfrey LM(2) = 6.8412, p=0.0327 -> re...",False
2,2,Breusch-Pagan LM,12.764162,13,0.466189,fail to reject H0 (p >= 0.05),"Breusch-Pagan LM = 12.7642 (df=13), p=0.4662 -...",False
3,3,Jarque-Bera,0.337180,2,0.844855,fail to reject H0 (p >= 0.05),"JB = 0.3372 (skew=-0.2439, kurtosis=2.9895), p...",True
4,4,Ramsey RESET,1.682198,"(2,18)",0.213913,fail to reject H0 (p >= 0.05),"RESET F(2,18) = 1.6822, p=0.2139 -> no evidenc...",True
5,5a,ARDL bounds F-test (restated from step vi),2.365753,"k=6, Case 3",NaN,INCONCLUSIVE at 5% (F-stat between bounds) -- ...,"Bounds F-stat = 2.3658 vs. 5% bounds [2.328, 3...",False
6,5b,CUSUM / CUSUMSQ stability (Brown-Durbin-Evans),NaN,NaN,NaN,skipped -- infeasible at the standard skip=k; ...,The SHOCK dummy is 0 for the first 18 of 34 us...,False


## Decisions & flags (explicit recap)

- **Jarque-Bera and Ramsey RESET are additions beyond the literal Ch. 3.7 methodology spec**,
  flagged in Steps 3 and 4 above and again here, for both Model A and Model B -- justified as
  standard practice and trivial to reproduce in EViews, not part of the thesis's stated
  methodology.
- **DW vs. Breusch-Godfrey is model-specific, not a free choice.** Model A (no lagged dependent
  variable) uses DW; Model B (lagged-dependent-variable-by-construction, via `ERI.L1`) uses
  Breusch-Godfrey at lag 1, matching the model's own `(p,q)=(1,1)` structure. Neither test is run
  on the other model.
- **Model A's DW is tested against exact Savin-White (1977) 5% bounds (n=35, k=6: dL=1.160,
  dU=1.803), not a rough "close to 2" heuristic.** DW=1.5566 falls strictly between dL and dU --
  **inconclusive**, not "no autocorrelation." See the Step 1 markdown above for the full citation
  (Savin & White, *Econometrica* 1977, table reproduced by the University of York's historical
  statistics archive).
- **Model B's Breusch-Godfrey borderline result (lag 1, p=0.0685) is NOT robust to lag choice.**
  Re-run at lag 2 as an explicit sensitivity check: LM(2) is significant at 5%. The lag-1 test is
  still the primary result (it matches the model's own `(p,q)=(1,1)` structure), but the lag-2
  result is reported alongside it rather than treated as a footnote, since it changes the
  substantive read on whether Model B's residuals are autocorrelated.
- **CUSUM/CUSUMSQ were attempted, not silently omitted, and came back infeasible.** The standard
  Brown-Durbin-Evans recursive-residual approach requires a non-singular initial window of `k`
  observations; the SHOCK dummy (0 for 1990-2007, the first 18 of 34 usable observations) makes
  the standard `skip=k=14` window rank-deficient. The smallest feasible `skip` (if any) and the
  resulting recursive-residual count are reported in `cusum_cusumsq_model_b_status.csv` --
  forcing a non-standard skip was rejected as producing too thin a path to be a meaningful
  stability check, per the plan's explicit "skip rather than force an unreliable result"
  instruction.
- **Newey-West `maxlags`** uses the stated default `floor(4*(T/100)^(2/9))`; with T=35 (Model A,
  first-differenced OLS uses T=34) this evaluates to 3 for both, and to 3 again for Model B's own
  T=34 (see Step 6/6b output).
- **HAC-robust SEs for Model A are reported side by side with the original SEs unconditionally**
  (`model_a_hac_coefficients.csv`), not gated behind a strict rejection -- the DW bounds test came
  back inconclusive (not clean), which alone is reason enough for a precautionary robustness
  cross-check, independent of whether Breusch-Pagan also flagged a problem. The original SEs are
  not replaced; both columns are kept for the EViews comparison per the plan.
- **DIVP's original-vs-HAC p-value shift is stated explicitly (Step 6):** original p=0.0444
  (already significant at 5% under the non-robust SEs), HAC p=0.0355 -- a modest tightening, not a
  crossing of the significance threshold. Reported precisely rather than left implied.
- **HAC remediation is scoped to Model A only per the plan's literal text**, but extended to
  Model B's long-run and short-run coefficients as well (Step 6b), by explicit request, given
  Step 1's lag-2 Breusch-Godfrey result (significant at 5%). This goes beyond the plan's literal
  scope and is flagged as such in Step 6b's markdown, with the lagged-dependent-variable caveat
  stated explicitly (Newey-West's textbook asymptotics assume strictly exogenous regressors;
  Model B's `ERI.L1` term is a lagged dependent variable by construction).
- **The first-differenced OLS is diagnosed as its own model** (DW/BP/JB/RESET, `n=34, k=6`),
  per this plan's own "Decisions & flags" instruction that Branch B triggers diagnostics on it too
  -- not skipped as "just a comparison model." Its Durbin-Watson (2.3812) is **also inconclusive**
  at 5% against the Savin-White n=34, k=6 bounds (dL=1.144, dU=1.807: DW falls in the negative-
  autocorrelation-side gray zone, `4-dU=2.193 < DW < 4-dL=2.856`) -- flagged here rather than
  silently treated as clean.
- **The first-differenced OLS was subsequently redesignated as the primary model for H1/H2**
  (`outputs/modeling_path_decision.csv` addendum; `research_plan.md`'s 2026-07-17 Update note),
  based on the ARDL bounds test not confirming cointegration at either specification (step vi)
  and Model B's lag-2 Breusch-Godfrey result (this notebook, above). Because it is now the
  primary model and its own DW came back inconclusive rather than clean, the same HAC precaution
  applied to Model A is applied to it too (Step 6c, `model_diff_hac_coefficients.csv`) --
  unconditionally, not gated behind a strict rejection, for the same reason as Model A.
- **DIVP and DIVM under HAC (Step 6c), reported precisely rather than left implied:** DIVP
  original p=0.0995 (10%-level only) -> HAC p=0.0573 -- **strengthens**, still short of 5% but
  materially closer. DIVM original p=0.1904 (not significant) -> HAC p=0.0915 -- **strengthens
  into 10%-level significance under HAC**, though its coefficient is negative (contrary to H2's
  hypothesized positive direction), consistent with the plan's own instruction to report a
  negative-and-significant finding rather than treat it as an error.

## Conclusion

**Model A -- static OLS on levels.** Breusch-Pagan, Jarque-Bera and Ramsey RESET all came back
clean at 5%. **Durbin-Watson is inconclusive, not clean**, when tested against the exact
Savin-White (1977) 5% bounds for n=35, k=6 (dL=1.160, dU=1.803): DW=1.5566 falls strictly between
them, so the bounds test cannot rule out positive autocorrelation. See
`diagnostics_model_a_ols.csv` for the exact numbers (regenerated on every run; do not hardcode
them here). Because of that inconclusive DW read, **Newey-West HAC-robust standard errors are
computed and reported side by side with the original OLS SEs unconditionally**
(`model_a_hac_coefficients.csv`, `maxlags=3` via `floor(4*(35/100)^(2/9))`) -- a precautionary
robustness check and EViews cross-check, not a replacement for the original SEs.

**Model B -- capped ARDL(1,1) / UECM (carried forward from step vi).** Breusch-Pagan, Jarque-Bera
and Ramsey RESET all came back clean at 5%. **Breusch-Godfrey is sensitive to lag order**: at lag
1 (the model's own `(p,q)=(1,1)` structure) it is borderline, not significant at 5%
(p=0.0685); re-run at lag 2 as an explicit sensitivity check, it **is** significant at 5%
(p=0.0327) -- reported as a genuine finding, not smoothed into "no autocorrelation." The Step B8
bounds F-test is restated here (still inconclusive at 5%, per step vi). CUSUM/CUSUMSQ
parameter-stability tests were attempted and found infeasible given the sample's SHOCK-dummy
structure -- skipped and documented in `cusum_cusumsq_model_b_status.csv`, not silently omitted.
Because of the lag-2 Breusch-Godfrey result, **HAC-robust SEs were also computed for Model B's
long-run and short-run coefficients** (Step 6b, `ardl_capped_1_1_long_run_hac_comparison.csv` /
`ardl_capped_1_1_short_run_hac_comparison.csv`), extending beyond the plan's literal Model-A-only
scope by explicit request.

**First-differenced OLS -- redesignated the primary model for H1/H2 inference.** Diagnosed with
the same DW/BP/JB/RESET battery as Model A (`diagnostics_model_diff_ols.csv`) -- Breusch-Pagan,
Jarque-Bera and Ramsey RESET all clean at 5%; Durbin-Watson (2.3812) is **inconclusive** at 5%
against the exact Savin-White n=34, k=6 bounds, on the negative-autocorrelation side. Given the
ARDL bounds test's failure to confirm cointegration (either specification, step vi) and Model B's
lag-2 Breusch-Godfrey result (above), **this model is now the primary model for H1/H2 inference**
(`outputs/modeling_path_decision.csv` addendum, `research_plan.md` 2026-07-17 Update) -- ARDL(1,1)
is retained and reported as secondary/exploratory. Because its own DW is inconclusive, HAC-robust
SEs are computed and reported unconditionally (Step 6c, `model_diff_hac_coefficients.csv`):
**DIVP's marginal 10%-level result strengthens under HAC** (p=0.0995 -> p=0.0573), and **DIVM
strengthens into 10%-level significance under HAC** (p=0.1904 -> p=0.0915) with a negative sign,
contrary to H2's hypothesized direction.

**Side-by-side tables:** `diagnostics_model_a_ols.csv`, `diagnostics_model_b_ardl_capped.csv`, and
`diagnostics_model_diff_ols.csv` each list every test run, in the exact sequence specified by the
plan, with statistic, p-value (where applicable), and a plain-language interpretation per row.

### Definition of done

- [x] All diagnostic tests specified in the plan (DW/Breusch-Godfrey, Breusch-Pagan, Jarque-Bera,
      Ramsey RESET, plus the ARDL-specific bounds restatement and CUSUM/CUSUMSQ attempt) run in
      the specified order, on every model estimated in step vi that the plan scopes them to --
      including the first-differenced OLS, per the plan's own Branch-B diagnostics instruction.
- [x] Each test's statistic, p-value, and plain-language interpretation reported in the per-model
      diagnostics CSVs.
- [x] Jarque-Bera and RESET explicitly flagged as additions beyond Ch. 3.7, every time they are
      presented (Steps 3-4 markdown and again in Decisions & flags), not just once in passing.
- [x] CUSUM/CUSUMSQ infeasibility is a data-driven, documented judgment call (SHOCK-dummy-induced
      singularity at the standard recursive-residual skip), not a silent omission.
- [x] Model A's and the first-differenced OLS's DW checked against exact Savin-White critical
      values (cited), not a rough heuristic -- both found inconclusive at 5%, not "no
      autocorrelation."
- [x] Newey-West HAC-robust SEs computed and reported side by side with the original OLS SEs for
      Model A, unconditionally, with the exact `maxlags=3` documented and its formula stated; the
      exact DIVP original-vs-HAC p-value shift stated explicitly.
- [x] Model B's borderline Breusch-Godfrey result cross-checked at a second lag order (lag 2);
      the sensitivity to lag choice is reported explicitly, not hidden behind the lag-1 number.
- [x] HAC-robust SEs extended to Model B's long-run and short-run coefficients, side by side with
      the originals, verified via an exact reproduction of step vi's saved delta-method SEs before
      the HAC column is trusted.
- [x] First-differenced OLS redesignated the primary model for H1/H2 (plan-of-record files
      updated: `modeling_path_decision.csv` addendum, `research_plan.md` Update note); HAC-robust
      SEs computed for it unconditionally given its own inconclusive DW, with DIVP/DIVM's exact
      original-vs-HAC p-values reported explicitly.